In [2]:
!pip install geopandas 

In [3]:
import sys
import herbie
import xarray as xr
import zarr as za
from herbie import Herbie
H = Herbie('2024-01-01', model='urma')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import geopandas as gpd
print("\n✓ STACK READY!!!!")
from shapely.geometry import Point

✅ Found ┊ model=urma ┊ product=anl ┊ 2024-Jan-01 00:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ None

✓ STACK READY!!!!


In [4]:
YEAR = '2024_full'
INPUT = f'01_URMA_RAW/URMA_RAW_{YEAR}.zarr'
OUTPUT = f'02_URMA_COUNTY/URMA_COUNTY_{YEAR}.zarr'

In [5]:
import zarr

path = f"{INPUT}"
z = zarr.open(path, mode="r")

def try_read(name, slicer=None):
    arr = z[name]
    try:
        if slicer is None:
            _ = arr[:]
        else:
            _ = arr[slicer]
        print(f"✅ {name} readable")
        return True
    except Exception as e:
        print(f"💥 {name} FAILED: {type(e).__name__}: {e}")
        return False

# Start with date (most likely culprit)
try_read("date")

# Then spot-check a couple variables (small slices so it’s quick)
for v in ["t2m_max_k", "cloud_cover_pct", "dpt_morning_k"]:
    try_read(v, (0, slice(None), slice(None)))


✅ date readable
✅ t2m_max_k readable
✅ cloud_cover_pct readable
✅ dpt_morning_k readable


In [6]:
directory = f"{INPUT}"
ds = xr.open_zarr(INPUT, consolidated=False, create_default_indexes=False)
ds

<xarray.Dataset> Size: 833MB
Dimensions:                (date: 106, y: 535, x: 457)
Coordinates:
    date                   (date) datetime64[ns] 848B dask.array<chunksize=(1,), meta=np.ndarray>
    atmosphereSingleLayer  float64 8B ...
    longitude              (y, x) float64 2MB dask.array<chunksize=(535, 457), meta=np.ndarray>
    latitude               (y, x) float64 2MB dask.array<chunksize=(535, 457), meta=np.ndarray>
Dimensions without coordinates: y, x
Data variables:
    t2m_min_k              (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    spfh_peak_kgkg         (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    t2m_max_k              (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    cloud_cover_pct        (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    wind_low_ms            (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    gribfile_projection    float64 8B ...
    dpt_afternoon_k        (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    dpt_morning_k          (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    wind_peak_ms           (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>

In [7]:
ds.date.min().compute()

<xarray.DataArray 'date' ()> Size: 8B
array('2024-01-01T00:00:00.000000000', dtype='datetime64[ns]')
Coordinates:
    atmosphereSingleLayer  float64 8B 0.0

In [8]:
ds.date.max().compute()

<xarray.DataArray 'date' ()> Size: 8B
array('2024-04-16T00:00:00.000000000', dtype='datetime64[ns]')
Coordinates:
    atmosphereSingleLayer  float64 8B 0.0

In [9]:
ds.date.min()

<xarray.DataArray 'date' ()> Size: 8B
dask.array<min-aggregate, shape=(), dtype=datetime64[ns], chunksize=(), chunktype=numpy.ndarray>
Coordinates:
    atmosphereSingleLayer  float64 8B ...

In [10]:
### Pull in TIGER counties
counties = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip"
)

# Filter to California
counties = counties[counties["STATEFP"] == "06"]

In [11]:
n_counties = counties['NAME'].nunique()
print(f'Contains = {n_counties}. \nShould contain 58 unique counties.')

Contains = 58. 
Should contain 58 unique counties.


In [12]:
# Convert URMA to longitude coordinates
ds_fixed = ds.copy()
ds_fixed = ds_fixed.assign_coords(
    longitude=(ds.longitude.dims, np.where(ds.longitude > 180, 
                                          ds.longitude - 360, 
                                          ds.longitude))
)

# Verify the fix
print("Fixed longitude range:", ds_fixed.longitude.values.min(), "to", ds_fixed.longitude.values.max())
# Should now show roughly -127 to -112°W

Fixed longitude range: -127.48243310481524 to -112.20237479820963


In [13]:
print("Latitude range:", ds_fixed.latitude.values.min(), "to", ds_fixed.latitude.values.max())
print("Longitude range:", ds_fixed.longitude.values.min(), "to", ds_fixed.longitude.values.max())

Latitude range: 30.399658978941538 to 43.89945484380344
Longitude range: -127.48243310481524 to -112.20237479820963


In [14]:
ds=ds_fixed

In [15]:
counties.shape

(58, 19)

In [16]:
## REMOVE UNNECESSARY VARIABLES
ds = ds.drop_vars(["gribfile_projection","atmosphereSingleLayer"])

In [17]:
## UPDATE URMA TO PROPER CRS COORDINATE SYSTEM

In [18]:
lat = ds.latitude.values
lon = ds.longitude.values

points = pd.DataFrame({
    "lat": lat.flatten(),
    "lon": lon.flatten()
})

gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points.lon, points.lat),
    crs="EPSG:4326"
)

In [19]:
counties = counties.to_crs("EPSG:4326")
gdf_points = gpd.sjoin(
    gdf_points,
    counties[["NAME", "geometry"]],
    how="left",
    predicate="within"
)

In [20]:
## RUN THE MASK TO AD COUNTIES 

In [21]:
county_mask = (
    gdf_points["NAME"]
      .fillna("Outside_CA")
      .to_numpy(dtype="object")
      .reshape(lat.shape)
)

ds = ds.assign_coords(county=(("y","x"), county_mask))

In [22]:
# Make sure TIGER and URMA use the same coordinate system (WGS84)
if counties.crs != 'EPSG:4326':
    print("Converting county CRS to EPSG:4326...")
    ca_counties = ca_counties.to_crs('EPSG:4326')

# Check coordinate bounds
print(f"County bounds: {counties.total_bounds}")
print(f"URMA lat range: {ds.latitude.min().values:.3f} to {ds.latitude.max().values:.3f}")
print(f"URMA lon range: {ds.longitude.min().values:.3f} to {ds.longitude.max().values:.3f}")

County bounds: [-124.482003     32.529508   -114.13120908   42.00950534]
URMA lat range: 30.400 to 43.899
URMA lon range: -127.482 to -112.202


In [23]:
# TESTING NEW URMA
def test_point_mapping(lon, lat, counties_gdf):
    point = Point(lon, lat)
    for idx, county_row in counties_gdf.iterrows():
        if county_row.geometry.contains(point):
            return county_row['NAME']
    return 'Outside_CA'

# Test some known California points
test_points = [
    (-121.5, 38.5),  # Sacramento
    (-118.2, 34.1),  # Los Angeles
    (-122.4, 37.8),  # San Francisco
]

print("Testing point mapping:")
for lon, lat in test_points:
    county = test_point_mapping(lon, lat, counties)
    print(f"Point ({lon}, {lat}) -> {county}")

Testing point mapping:
Point (-121.5, 38.5) -> Sacramento
Point (-118.2, 34.1) -> Los Angeles
Point (-122.4, 37.8) -> San Francisco


In [24]:
# Make sure both use the same coordinate system (WGS84)
if counties.crs != 'EPSG:4326':
    print("Converting county CRS to EPSG:4326...")
    counties = counties.to_crs('EPSG:4326')

# Check coordinate bounds
print(f"County bounds: {counties.total_bounds}")
print(f"URMA lat range: {ds.latitude.min().values:.3f} to {ds.latitude.max().values:.3f}")
print(f"URMA lon range: {ds.longitude.min().values:.3f} to {ds.longitude.max().values:.3f}")

County bounds: [-124.482003     32.529508   -114.13120908   42.00950534]
URMA lat range: 30.400 to 43.899
URMA lon range: -127.482 to -112.202


In [25]:
def assign_counties_to_grid(ds, counties_gdf, county_column='NAME'):
    """
    Assign county names to each grid point in the dataset
    """
    print("Creating county assignments...")
    
    # Get the grid dimensions
    ny, nx = ds.latitude.shape
    
    # Initialize county array
    county_array = np.full((ny, nx), 'Outside_CA', dtype=object)
    
    # Process each grid point
    points_processed = 0
    ca_points_found = 0
    
    for i in range(ny):
        if i % 50 == 0:  # Progress update
            print(f"Processing row {i}/{ny}")
            
        for j in range(nx):
            lat = ds.latitude.values[i, j]
            lon = ds.longitude.values[i, j]
            
            # Create point
            point = Point(lon, lat)
            points_processed += 1
            
            # Check which county contains this point
            for idx, county_row in counties_gdf.iterrows():
                if county_row.geometry.contains(point):
                    county_array[i, j] = county_row[county_column]
                    ca_points_found += 1
                    break
    
    print(f"Processed {points_processed} points")
    print(f"Found {ca_points_found} points in California counties")
    
    return county_array

In [26]:
#ACTUALLY ADDING COUNTY INFO TO UMRA, TAKES A WHILE
county_assignments = assign_counties_to_grid(ds, counties)

# Check the results
unique_counties = np.unique(county_assignments)
print(f"Found {len(unique_counties)} unique county values:")
for county in unique_counties:
    count = (county_assignments == county).sum()
    print(f"  {county}: {count} grid points")

Creating county assignments...
Processing row 0/535
Processing row 50/535
Processing row 100/535
Processing row 150/535
Processing row 200/535
Processing row 250/535
Processing row 300/535
Processing row 350/535
Processing row 400/535
Processing row 450/535
Processing row 500/535
Processed 244495 points
Found 69035 points in California counties
Found 59 unique county values:
  Alameda: 346 grid points
  Alpine: 316 grid points
  Amador: 255 grid points
  Butte: 720 grid points
  Calaveras: 438 grid points
  Colusa: 494 grid points
  Contra Costa: 340 grid points
  Del Norte: 535 grid points
  El Dorado: 765 grid points
  Fresno: 2522 grid points
  Glenn: 569 grid points
  Humboldt: 1754 grid points
  Imperial: 1838 grid points
  Inyo: 4277 grid points
  Kern: 3387 grid points
  Kings: 580 grid points
  Lake: 570 grid points
  Lassen: 2053 grid points
  Los Angeles: 1960 grid points
  Madera: 907 grid points
  Marin: 349 grid points
  Mariposa: 618 grid points
  Mendocino: 1667 grid poi

In [27]:
# INCLUDE COUNTY AS A COORD
ds_with_counties = ds.assign_coords(
    county=(('y', 'x'), county_assignments)
)

# Verify it worked
print(f"Dataset now has county coordinate: {'county' in ds_with_counties.coords}")
print(f"County coordinate shape: {ds_with_counties.county.shape}")
print(f"Sample county values: {ds_with_counties.county.values.flat[:10]}")

Dataset now has county coordinate: True
County coordinate shape: (535, 457)
Sample county values: ['Outside_CA' 'Outside_CA' 'Outside_CA' 'Outside_CA' 'Outside_CA'
 'Outside_CA' 'Outside_CA' 'Outside_CA' 'Outside_CA' 'Outside_CA']


In [28]:
all_counties = ds_with_counties.county.values.flatten()
unique_counties, counts = np.unique(all_counties, return_counts=True)

print(f"Total unique values: {len(unique_counties)}")
print(f"California counties found: {len([c for c in unique_counties if c != 'Outside_CA'])} (SHOULD BE 58)")


Total unique values: 59
California counties found: 58 (SHOULD BE 58)


In [29]:
import numpy as np
import pandas as pd
import xarray as xr

ds = ds_with_counties

# grab county labels into memory once (535*457 ~ 245k, totally fine)
vals = ds["county"].values
flat = vals.ravel()

mask = pd.notna(flat)
flat_str = flat[mask].astype(str)

cats, inv = np.unique(flat_str, return_inverse=True)

codes = np.full(flat.shape, -1, dtype=np.int16)  # -1 = missing
codes[mask] = inv.astype(np.int16)
codes2d = codes.reshape(vals.shape)

# replace county strings with integer codes
ds2 = ds.drop_vars("county").assign_coords(
    county_code=(("y","x"), codes2d)
)

# store the lookup table so you can decode later
ds2["county_code"].attrs["categories"] = cats.tolist()
ds2["county_code"].attrs["missing_code"] = -1

ds2


<xarray.Dataset> Size: 834MB
Dimensions:          (date: 106, y: 535, x: 457)
Coordinates:
    date             (date) datetime64[ns] 848B dask.array<chunksize=(1,), meta=np.ndarray>
    longitude        (y, x) float64 2MB -124.1 -124.1 -124.1 ... -114.3 -114.2
    latitude         (y, x) float64 2MB dask.array<chunksize=(535, 457), meta=np.ndarray>
    county_code      (y, x) int16 489kB 30 30 30 30 30 30 ... 30 30 30 30 30 30
Dimensions without coordinates: y, x
Data variables:
    t2m_min_k        (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    spfh_peak_kgkg   (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    t2m_max_k        (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    cloud_cover_pct  (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    wind_low_ms      (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    dpt_afternoon_k  (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    dpt_morning_k    (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    wind_peak_ms     (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>

In [30]:
OUTPUT 

'02_URMA_COUNTY/URMA_COUNTY_2024_full.zarr'

## Write to Zarr

In [31]:
import os
os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)

# (Optional) make sure chunks are what you want (y/x could be smaller if desired)
# ds2 = ds2.chunk({"date": 1, "y": 200, "x": 200})

ds.to_zarr(OUTPUT, mode="w")   # overwrite / fresh write

/Users/amysteward/miniconda3/envs/herbie_z3/lib/python3.14/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [32]:
ds_out = xr.open_zarr(OUTPUT, consolidated=False)
ds_out

<xarray.Dataset> Size: 835MB
Dimensions:          (date: 106, y: 535, x: 457)
Coordinates:
  * date             (date) datetime64[ns] 848B 2024-01-01 ... 2024-04-16
    county           (y, x) object 2MB dask.array<chunksize=(535, 457), meta=np.ndarray>
    longitude        (y, x) float64 2MB dask.array<chunksize=(134, 229), meta=np.ndarray>
    latitude         (y, x) float64 2MB dask.array<chunksize=(535, 457), meta=np.ndarray>
Dimensions without coordinates: y, x
Data variables:
    wind_low_ms      (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    dpt_morning_k    (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    cloud_cover_pct  (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    t2m_min_k        (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    t2m_max_k        (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    spfh_peak_kgkg   (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    dpt_afternoon_k  (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    wind_peak_ms     (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>

In [33]:
ds

<xarray.Dataset> Size: 835MB
Dimensions:          (date: 106, y: 535, x: 457)
Coordinates:
    date             (date) datetime64[ns] 848B dask.array<chunksize=(1,), meta=np.ndarray>
    longitude        (y, x) float64 2MB -124.1 -124.1 -124.1 ... -114.3 -114.2
    latitude         (y, x) float64 2MB dask.array<chunksize=(535, 457), meta=np.ndarray>
    county           (y, x) object 2MB 'Outside_CA' ... 'Outside_CA'
Dimensions without coordinates: y, x
Data variables:
    t2m_min_k        (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    spfh_peak_kgkg   (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    t2m_max_k        (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    cloud_cover_pct  (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    wind_low_ms      (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    dpt_afternoon_k  (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    dpt_morning_k    (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    wind_peak_ms     (date, y, x) float32 104MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>

In [34]:
## STOP STOP

In [35]:

county_counts = dict(zip(unique_counties, counts))
sorted_counties = sorted(county_counts.items(), key=lambda x: x[1], reverse=True)

print("\nTop 15 areas by grid point count:")
for county, count in sorted_counties[:15]:
    print(f"  {county}: {count:,} points")

# Verify California counties are there
ca_counties = [c for c in unique_counties if c != 'Outside_CA']
print(f"\nSample California counties found:")
for county in sorted(ca_counties)[:10]:
    print(f"  {county}")


Top 15 areas by grid point count:
  Outside_CA: 175,460 points
  San Bernardino: 8,326 points
  Inyo: 4,277 points
  Kern: 3,387 points
  Riverside: 3,005 points
  Siskiyou: 2,787 points
  Fresno: 2,522 points
  Lassen: 2,053 points
  Tulare: 2,022 points
  Los Angeles: 1,960 points
  San Diego: 1,856 points
  Modoc: 1,843 points
  Imperial: 1,838 points
  Humboldt: 1,754 points
  Shasta: 1,669 points

Sample California counties found:
  Alameda
  Alpine
  Amador
  Butte
  Calaveras
  Colusa
  Contra Costa
  Del Norte
  El Dorado
  Fresno


In [36]:
#QC Zarr

In [37]:
## OPTIONAL  QC

In [38]:
# Test some points that should definitely be in California counties FROM TIGER
test_points = [
    (-121.5, 38.5),  # Sacramento area
    (-118.2, 34.1),  # LA area  
    (-122.4, 37.8),  # SF area
    (-119.8, 36.7),  # Fresno area
]

print("Testing known California points:")
for lon, lat in test_points:
    point = Point(lon, lat)
    found_county = None
    
    for idx, county_row in counties.iterrows():
        if county_row.geometry.contains(point):
            found_county = county_row['NAME']
            break
    
    if found_county:
        print(f"✓ Point ({lon}, {lat}) is in {found_county}")
    else:
        print(f"✗ Point ({lon}, {lat}) not found in any county")

# Also check what column contains the county names
print(f"\nAvailable columns: {counties.columns.tolist()}")
print(f"Sample county names from 'NAME' column: {counties['NAME'].head().tolist()}")

Testing known California points:
✓ Point (-121.5, 38.5) is in Sacramento
✓ Point (-118.2, 34.1) is in Los Angeles
✓ Point (-122.4, 37.8) is in San Francisco
✓ Point (-119.8, 36.7) is in Fresno

Available columns: ['STATEFP', 'COUNTYFP', 'COUNTYNS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'LSAD', 'CLASSFP', 'MTFCC', 'CSAFP', 'CBSAFP', 'METDIVFP', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry']
Sample county names from 'NAME' column: ['Sierra', 'Sacramento', 'Santa Barbara', 'Calaveras', 'Ventura']
